# 01 · Descarga y exploración inicial (OpenAlex — Computer Science)

**Proyecto:** predecir si un artículo de *Computer Science* llega al **top 10 % de citas** (y su número de citas), y medir el aporte de los **laboratorios privados / Big Tech** frente a otras empresas y a la academia.

Este notebook es la **Tarea 1**: sienta las bases de la descarga. No descarga todo el corpus; sólo valida la API, los filtros y baja una muestra pequeña.

### Qué hace, por sección
1. Carga de la clave desde `.env` y función `get()` con reintentos ante 429/5xx.
2. Subcampos de *Computer Science* (field id 17) con su id y `works_count`.
3. Conteos de prueba (`per_page=1`, `meta.count`): artículos CS por año 2014–2021, con empresa y sin empresa. **Se verifica cada filtro.**
4. Búsqueda de IDs de institución de Big Tech → `src/bigtech_institutions.csv` (para revisión manual).
5. Muestra de 500 artículos por cursor → `data/raw/sample_500.parquet` + inventario de columnas/nulos.

### Reglas que respeta este notebook
- **No** filtra por número de citas ni por revista.
- El JSON crudo se guarda en `data/raw`; la limpieza irá luego en `data/processed`.
- La reconstrucción del abstract desde `abstract_inverted_index` se hará **más adelante**.
- El muestreo definitivo (todos los que tengan empresa + ~2.000/año sin empresa, con `seed` fijo) **aún no se implementa**.

### ⚠️ Columnas que NO se usan como variable (fuga de información)
`cited_by_count`, `counts_by_year`, `fwci`, `citation_normalized_percentile`, `cited_by_percentile_year`, `is_in_top_1_percent`, `related_works`, `updated_date`.

Son el objetivo o derivan del futuro del artículo. Se descargan igualmente (para construir la etiqueta top‑10 %), pero quedan marcadas y **no** entran como predictores.

## 1 · Clave de API y función `get()` con reintentos

Cargamos `OPENALEX_API_KEY` desde `.env` (**nunca** se imprime). La función `get()`:
- añade la clave si existe, y un `mailto` opcional (`OPENALEX_MAILTO`) para el *polite pool*;
- reintenta ante **429** (rate limit) y **5xx**, respetando la cabecera `Retry-After` si viene, con *backoff* exponencial;
- ante otros errores 4xx lanza el error con el cuerpo de la respuesta (para depurar sintaxis de filtros).

> OpenAlex funciona **sin clave** a través del *pool* común gratuito; la clave sólo da acceso al *pool* autenticado/premium. Si `.env` está vacío, el notebook sigue funcionando (más lento).

In [ ]:
import os
import time
import json
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

# --- Rutas del proyecto (el notebook vive en notebooks/) ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SRC_DIR = PROJECT_ROOT / "src"
for d in (RAW_DIR, PROCESSED_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Clave (nunca se imprime) ---
load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.getenv("OPENALEX_API_KEY")
MAILTO = os.getenv("OPENALEX_MAILTO")  # opcional, recomendado por OpenAlex (polite pool)

print("API key cargada:", "sí" if API_KEY else "NO (se usa el pool común gratuito)")
print("mailto configurado:", "sí" if MAILTO else "no")

BASE = "https://api.openalex.org"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "ensae-asl-project/1.0 (academic use)"})

In [ ]:
class OpenAlexError(RuntimeError):
    pass


def get(path, params=None, max_retries=6, timeout=60):
    """GET a OpenAlex con reintentos ante 429/5xx.

    path: '/works', '/institutions', '/subfields'... o una URL completa.
    Devuelve el JSON decodificado. La clave/mailto se añaden aquí y no se registran.
    """
    params = dict(params or {})
    if API_KEY:
        params["api_key"] = API_KEY
    if MAILTO:
        params["mailto"] = MAILTO
    url = path if path.startswith("http") else f"{BASE}{path}"

    backoff = 1.0
    last_body = ""
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
        except requests.RequestException as exc:
            if attempt == max_retries:
                raise OpenAlexError(f"Fallo de red tras {max_retries} intentos: {exc}") from exc
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
            continue

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429 or 500 <= r.status_code < 600:
            # pausa ante rate-limit / errores de servidor
            wait = r.headers.get("Retry-After")
            wait = float(wait) if wait and wait.replace('.', '', 1).isdigit() else backoff
            last_body = r.text[:300]
            if attempt == max_retries:
                raise OpenAlexError(f"{r.status_code} tras {max_retries} intentos. Cuerpo: {last_body}")
            time.sleep(wait)
            backoff = min(backoff * 2, 60)
            continue

        # 4xx distinto de 429: normalmente sintaxis de filtro mal formada -> mostrar cuerpo
        raise OpenAlexError(f"HTTP {r.status_code} en {path}. Cuerpo: {r.text[:300]}")

    raise OpenAlexError("max_retries agotado sin respuesta")


def count_for(filter_str):
    """Atajo: devuelve meta.count de /works para un filtro, con per_page=1."""
    j = get("/works", {"filter": filter_str, "per_page": 1})
    return j["meta"]["count"]


# Columnas marcadas como fuga de información (NO usar como variable predictora)
LEAKAGE_COLS = [
    "cited_by_count",
    "counts_by_year",
    "fwci",
    "citation_normalized_percentile",
    "cited_by_percentile_year",
    "is_in_top_1_percent",
    "related_works",
    "updated_date",
]
print("get() listo. Columnas marcadas como fuga:", len(LEAKAGE_COLS))

## 2 · Subcampos de *Computer Science* (field id 17)

OpenAlex organiza los temas en jerarquía **domain → field → subfield → topic**. *Computer Science* es el field **17**. Listamos sus subcampos con `id` y `works_count` para entender la composición interna del campo (NLP, visión, teoría, etc.).

In [ ]:
sub = get("/subfields", {"filter": "field.id:17", "per_page": 200})
print("Subcampos de CS:", sub["meta"]["count"])

subfields_df = pd.DataFrame(
    [
        {
            "subfield_id": s["id"].rsplit("/", 1)[-1],
            "display_name": s["display_name"],
            "works_count": s.get("works_count"),
        }
        for s in sub["results"]
    ]
).sort_values("works_count", ascending=False, ignore_index=True)

subfields_df

## 3 · Conteos de prueba (`per_page=1`, `meta.count`) + verificación

Tres bloques, todos restringidos a **field 17 + `type:article`**:
1. artículos CS por año **2014–2021**;
2. cuántos tienen alguna institución de tipo **company** (`institutions.type:company`);
3. cuántos **sin empresa** (= total del año − con empresa).

**Verificación:** antes de seguir comprobamos que (a) el total de un año es *muy* menor que todo OpenAlex (el filtro de campo realmente filtra), y (b) `con empresa < total`. Si el filtro de campo o de empresa falla, el notebook prueba una sintaxis alternativa y avisa cuál funcionó.

In [ ]:
# --- Elegir sintaxis de filtro de CAMPO que realmente filtre ---
# OpenAlex ha aceptado distintas claves según versión; probamos candidatas y nos
# quedamos con la primera cuyo conteo sea razonable (menor que todo OpenAlex).
TOTAL_OPENALEX = get("/works", {"per_page": 1})["meta"]["count"]
print("Works totales en OpenAlex:", f"{TOTAL_OPENALEX:,}")

CS_FIELD_CANDIDATES = [
    "primary_topic.field.id:17",
    "primary_topic.field.id:fields/17",
    "topics.field.id:17",
]

CS_FIELD_FILTER = None
for cand in CS_FIELD_CANDIDATES:
    try:
        c = count_for(f"{cand},type:article,publication_year:2018")
    except OpenAlexError as e:
        print(f"  [x] {cand:38s} -> ERROR: {str(e)[:80]}")
        continue
    ratio = c / TOTAL_OPENALEX
    ok = 0 < c < TOTAL_OPENALEX * 0.5  # debe filtrar de verdad
    print(f"  [{'ok' if ok else '??'}] {cand:38s} -> {c:,} artículos CS/2018 (ratio {ratio:.4f})")
    if ok and CS_FIELD_FILTER is None:
        CS_FIELD_FILTER = cand

assert CS_FIELD_FILTER, "Ninguna sintaxis de filtro de campo funcionó; revisar manualmente."
print("\n>>> Sintaxis de CAMPO adoptada:", CS_FIELD_FILTER)

In [ ]:
# --- Elegir sintaxis de filtro de EMPRESA que funcione ---
base_2018 = f"{CS_FIELD_FILTER},type:article,publication_year:2018"
total_2018 = count_for(base_2018)

COMPANY_CANDIDATES = [
    "institutions.type:company",
    "authorships.institutions.type:company",
]

COMPANY_FILTER = None
for cand in COMPANY_CANDIDATES:
    try:
        c = count_for(f"{base_2018},{cand}")
    except OpenAlexError as e:
        print(f"  [x] {cand:42s} -> ERROR: {str(e)[:80]}")
        continue
    ok = 0 < c < total_2018  # con empresa debe ser un subconjunto propio
    print(f"  [{'ok' if ok else '??'}] {cand:42s} -> {c:,} con empresa / {total_2018:,} total 2018")
    if ok and COMPANY_FILTER is None:
        COMPANY_FILTER = cand

assert COMPANY_FILTER, "Ninguna sintaxis de filtro de empresa funcionó; revisar manualmente."
print("\n>>> Sintaxis de EMPRESA adoptada:", COMPANY_FILTER)

In [ ]:
# --- Conteos por año 2014-2021 ---
YEARS = list(range(2014, 2022))
rows = []
for y in tqdm(YEARS, desc="conteos por año"):
    base = f"{CS_FIELD_FILTER},type:article,publication_year:{y}"
    total = count_for(base)
    with_company = count_for(f"{base},{COMPANY_FILTER}")
    rows.append(
        {
            "year": y,
            "cs_articles": total,
            "with_company": with_company,
            "without_company": total - with_company,
            "pct_company": round(100 * with_company / total, 3) if total else None,
        }
    )

counts_df = pd.DataFrame(rows)
counts_df.loc["TOTAL"] = {
    "year": "2014-2021",
    "cs_articles": counts_df["cs_articles"].sum(),
    "with_company": counts_df["with_company"].sum(),
    "without_company": counts_df["without_company"].sum(),
    "pct_company": round(100 * counts_df["with_company"].sum() / counts_df["cs_articles"].sum(), 3),
}
counts_df

In [ ]:
# --- Verificaciones explícitas (sanity checks) ---
problems = []
for _, r in counts_df.iloc[:-1].iterrows():
    if not (r["with_company"] < r["cs_articles"]):
        problems.append(f"{r['year']}: con empresa ({r['with_company']}) no es < total ({r['cs_articles']})")
    if r["cs_articles"] >= TOTAL_OPENALEX * 0.5:
        problems.append(f"{r['year']}: total sospechosamente alto (¿el filtro de campo no filtra?)")
    if not (0 < r['pct_company'] < 50):
        problems.append(f"{r['year']}: %empresa fuera de rango plausible ({r['pct_company']}%)")

if problems:
    print("⚠️ PROBLEMAS detectados:")
    for p in problems:
        print("  -", p)
else:
    print("✅ Verificación OK: los filtros devuelven conteos razonables (campo filtra, empresa es subconjunto, %empresa plausible).")

# Guardamos los conteos para referencia (no es dato crudo de works)
counts_df.to_csv(PROCESSED_DIR / "counts_cs_by_year.csv", index=False)
subfields_df.to_csv(PROCESSED_DIR / "cs_subfields.csv", index=False)
print("\nGuardado: data/processed/counts_cs_by_year.csv y cs_subfields.csv")

## 4 · Búsqueda de IDs de institución de Big Tech → CSV para revisión manual

Buscamos con `/institutions?search=` cada nombre de la lista y guardamos **los primeros candidatos** de cada búsqueda en `src/bigtech_institutions.csv`.

**No se decide nada aquí.** Los casos ambiguos (DeepMind vs Google, Google Brain, filiales como Google Research / Google DeepMind, Microsoft Research vs Microsoft) se dejan para que los revises **a mano**: hay una columna `keep` vacía que rellenarás tú (1/0) y una columna `notes`. Se incluyen `type`, `country_code` y `works_count` para ayudar a desambiguar.

In [ ]:
BIGTECH_QUERIES = [
    "Google", "Google Research", "Google Brain", "DeepMind", "Google DeepMind",
    "Microsoft", "Microsoft Research",
    "Meta", "Meta AI", "Facebook", "Facebook AI Research",
    "OpenAI", "Anthropic",
    "Amazon", "Amazon Web Services",
    "IBM", "IBM Research",
    "Nvidia", "Apple",
    "Intel", "Adobe", "Samsung", "Huawei", "Tencent", "Alibaba", "Baidu",
    "Salesforce", "Uber", "Netflix", "Qualcomm", "Sony", "Yahoo",
]

rows = []
for q in tqdm(BIGTECH_QUERIES, desc="búsqueda instituciones"):
    res = get("/institutions", {"search": q, "per_page": 5})
    for rank, inst in enumerate(res["results"], start=1):
        rows.append(
            {
                "query": q,
                "rank": rank,
                "institution_id": inst["id"].rsplit("/", 1)[-1],
                "display_name": inst["display_name"],
                "type": inst.get("type"),
                "country_code": inst.get("country_code"),
                "works_count": inst.get("works_count"),
                "homepage": (inst.get("homepage_url") or ""),
                "ror": inst.get("ror", ""),
                "keep": "",   # <- rellenar a mano: 1 = sí, 0 = no
                "notes": "",  # <- anotaciones de desambiguación
            }
        )

bigtech_df = pd.DataFrame(rows)
out_csv = SRC_DIR / "bigtech_institutions.csv"
bigtech_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Guardado {len(bigtech_df)} candidatos en {out_csv.relative_to(PROJECT_ROOT)}")
print("→ Revísalo a mano y rellena la columna 'keep'. Los casos ambiguos quedan a tu criterio.")
bigtech_df.head(20)

## 5 · Muestra de 500 artículos (paginación por cursor) + inventario

Descargamos **500 artículos CS** (`type:article`, años 2014–2021) con **paginación por cursor** y `per_page=100`. Sin filtrar por citas ni revista. Guardamos:
- el **JSON crudo** en `data/raw/sample_500.json` (regla: el crudo siempre va a `data/raw`);
- una versión tabular mínima en `data/raw/sample_500.parquet`.

Luego hacemos un **inventario de columnas y nulos**. La reconstrucción del abstract y la limpieza se harán después.

In [ ]:
def fetch_cursor(filter_str, n_target, per_page=100, select=None):
    """Descarga hasta n_target works por paginación de cursor. Devuelve lista de dicts crudos."""
    out = []
    cursor = "*"
    pbar = tqdm(total=n_target, desc="descarga muestra")
    while cursor and len(out) < n_target:
        params = {"filter": filter_str, "per_page": per_page, "cursor": cursor}
        if select:
            params["select"] = ",".join(select)
        page = get("/works", params)
        results = page["results"]
        if not results:
            break
        out.extend(results)
        pbar.update(len(results))
        cursor = page["meta"].get("next_cursor")
    pbar.close()
    return out[:n_target]


SAMPLE_FILTER = f"{CS_FIELD_FILTER},type:article,publication_year:2014-2021"
sample_raw = fetch_cursor(SAMPLE_FILTER, n_target=500, per_page=100)
print("Descargados:", len(sample_raw), "artículos")

# JSON crudo -> data/raw
with open(RAW_DIR / "sample_500.json", "w", encoding="utf-8") as f:
    json.dump(sample_raw, f, ensure_ascii=False)
print("JSON crudo guardado en data/raw/sample_500.json")

In [ ]:
# Tabular mínima para inventario (sin aplanar todo; eso viene en la limpieza).
# json_normalize deja las columnas anidadas como objetos; suficiente para ver columnas/nulos.
sample_df = pd.json_normalize(sample_raw, max_level=1)

# parquet necesita columnas serializables: convertimos objetos complejos a JSON string.
def _safe(v):
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False)
    return v

sample_flat = sample_df.applymap(_safe)
sample_flat.to_parquet(RAW_DIR / "sample_500.parquet", index=False)
print("Parquet guardado:", RAW_DIR / "sample_500.parquet", "| shape:", sample_flat.shape)

In [ ]:
# --- Inventario de columnas y nulos ---
inv = pd.DataFrame(
    {
        "dtype": sample_df.dtypes.astype(str),
        "n_null": sample_df.isna().sum(),
        "pct_null": (100 * sample_df.isna().mean()).round(1),
        "ejemplo": [sample_df[c].dropna().iloc[0] if sample_df[c].notna().any() else None for c in sample_df.columns],
    }
)
inv["LEAKAGE"] = inv.index.to_series().apply(
    lambda c: "⚠️ NO USAR" if any(c == k or c.startswith(k + ".") for k in LEAKAGE_COLS) else ""
)
inv = inv.sort_values(["LEAKAGE", "pct_null"], ascending=[False, True])
print("Columnas:", len(inv), "| marcadas como fuga presentes:", (inv['LEAKAGE'] != '').sum())
inv

## Siguientes pasos (NO en este notebook)

- **Revisar a mano** `src/bigtech_institutions.csv` y rellenar `keep` (sobre todo DeepMind/Google, filiales).
- Muestreo definitivo: **todos** los artículos con empresa + **~2.000/año** sin empresa, con `sample(random_state=SEED)` fijo y reproducible.
- Reconstrucción del abstract desde `abstract_inverted_index`.
- Limpieza → `data/processed`.
- Construcción de la etiqueta **top 10 %** de citas (usando `cited_by_count` sólo para la *etiqueta*, nunca como predictor), controlando por año/edad del artículo.

> Antes de lanzar la **descarga completa**: revisar límites de la API (ver la nota que imprime la celda siguiente) y confirmar.

In [ ]:
print(
    "NOTA sobre límites de OpenAlex (pool gratuito):\n"
    "  - Sin coste por llamada (API gratuita).\n"
    "  - Límite ~100.000 llamadas/día y ~10 req/s.\n"
    "  - Con per_page=100 y cursor, 1 llamada = 100 works -> ~10M works/día como techo teórico.\n"
    "  - La clave premium (api_key) sube límites; el mailto da acceso al 'polite pool'.\n"
    "NO lanzar la descarga completa sin confirmar el plan de muestreo y los límites."
)